# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
from pprint import pprint

# List all record sets and their fields
print("Available record sets:")
for rs in dataset.metadata.record_sets:
    print(f"- Record Set: {rs.name} (@id: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, @type: {field.data_type})")
    print()

# Store the record set IDs for use later
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
pprint(record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

For this dataset, we extract available tables (record sets) using their `@id`. All entity references use their Croissant `@id` field.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Sample columns in {record_set_id}: ", df.columns.tolist())
        display(df.head(2))
    else:
        print(f"No records found for {record_set_id}.")

# Choose the main record set for analysis (the one with most columns or expected data e.g. the protein table)
if len(dataframes) > 0:
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[1])
    print(f"Main analysis will use record set @id: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
    print(main_df.columns.tolist())
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Let's examine, filter, and transform a numeric field (e.g., `Coverage_percent` or `MW_kDa` if present) in the main record set.

All columns below are referenced by their exact Croissant field `@id`, ensuring traceability.

In [ ]:
import numpy as np
# Pick a numeric field by @id (update as needed based on overview above)
# Example field`@id`s, please update as appropriate for this dataset:

# Try common ones:
candidate_numeric_fields = [
    'Coverage_percent', 'MW_kDa', 'Unique_peptides', 'Peptide_counts',
    'NormalizedAbundance', 'pI_theoretical', 'pI_empirical',
    'Intensity_sample_A', 'Intensity_sample_B' # Replace with actual field IDs from the overview
]

print("Candidate numeric fields (check which exist):")
existing = []
for col in candidate_numeric_fields:
    if col in main_df.columns:
        print(f"  Column: {col}")
        existing.append(col)

# Choose one numeric field for demo:
if existing:
    numeric_field_id = existing[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    numeric_field_id = main_df.select_dtypes(include=[np.number]).columns[0]
    print(f"Fallback to numeric column: {numeric_field_id}")

# Filter by threshold
threshold = main_df[numeric_field_id].quantile(0.75)
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, e.g., 'SampleID' or similar
candidate_group_fields = ['Sample', 'SampleID', 'Condition', 'Group', 'Protein_Class']
group_field = None
for col in candidate_group_fields:
    if col in main_df.columns:
        group_field = col
        break
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"Grouped mean of {numeric_field_id} by {group_field} (top 5):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the main numeric field and comparisons by group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if available)
if group_field is not None:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

This notebook explored the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using `mlcroissant`.

- We loaded the dataset schema and records using the Croissant specification.
- The table structure was discovered and key numeric fields extracted by their Croissant `@id`.
- Typical data processing steps for normalization and aggregation were demonstrated.
- Data distributions and group differences (where available) were visualized.

For rigorous analyses, review the underlying metadata fields and schema to select fields and thresholds relevant to your scientific question. Continue with further statistical modeling or domain-specific exploration as desired.
